# Prefect

This is a tutorial to use the Prefect cluster from Jupyter, without Dask.

In [ ]:
import os
print(f"Internal Prefect server: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['PREFECT_PUBLIC']}/dashboard"
print(f"Public Prefect dashboard: {dashboard}")

In [ ]:
from resources.utils import *
# Init environment before running a demo notebook.
init_demo()

# In local mode, init the prefect blocks.
# NOTE: In the cluster, the blocks must be created only once by the admin.
await init_prefect_blocks()

from resources.utils import *  # reload the global vars again

In [ ]:
%%bash
prefect block ls

In [ ]:
# Other imports
import getpass
import json
import logging
import os
import prefect
from resources.my_shared_utils import get_ip_address

# NOTE: when deploying, the prefect flows and tasks must be implemented in a separate python module.
# We cannot implement them directly in jupyter cells.
import my_prefect

# Data to test the example flow
my_data = [
    "PrefectHQ/prefect",
    "pydantic/pydantic",
    "huggingface/transformers"
] * 5 # repeat the data

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

### Implement the `quickstart` tutorial
See: https://docs.prefect.io/v3/get-started/quickstart

When calling the flow as a normal python function, the flow and tasks are run by Prefect on your local client environment = your Jupyter or terminal.

This is the easiest way to test your Prefect code because the same environment, Python interpreter and files are shared between your client, flow and tasks. But this is less performant because your tasks are not distributed on the cluster.

In [ ]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

# Run the flow
my_prefect.flow_show_stars(my_data)

<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs above that your client IP address is also used by the Prefect flow and tasks.
  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.

### Run flows in local processes
See: https://docs.prefect.io/v3/deploy/run-flows-in-local-processes

Create a deployment for a flow by calling the `serve` method.

As for the quickstart above, the same environment, Python interpreter and files are shared between your client, flow and tasks.

In [ ]:
# Deploy the flow
task = my_prefect.hack_for_jupyter( # we need a hack to deploy from jupyter
    my_prefect.flow_show_stars.serve,
    name="serve-python",
    tags=["tutorial"],
)
serve_python = "flow-show-stars/serve-python"
await my_prefect.wait_for_deployment(serve_python)

In [ ]:
%%bash -s "$serve_python" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2"

In [ ]:
from prefect.settings import PREFECT_UI_URL
print(f"""
########
# NOTE #
########

Don't use the internal domain from the logs above: {PREFECT_UI_URL.value()!r}, use the public domain instead: {os.environ['PREFECT_PUBLIC']}
""")

In [ ]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

<div class="alert alert-info" role="alert">
Notes:

  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.
  1. Check in the run logs that the client IP address is also used by the Prefect flow and tasks.

### Deploy flows with Python

See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/deploy-via-python

Prefect offers a flexible way to deploy flows to dynamic infrastructure using the Python SDK. This approach allows you to target specific work pools and utilize dynamically provisioned infrastructure.

This is easier to deploy than with YAML (see next section) but less complete (e.g. cannot run additional scripts or pip install ...)

**Deploy the source code**

You want to deploy flows and tasks from you local source code... but this source code doesn't exist in the prefect worker container. So you need to store it somewhere and transfer it. 

We can use this project git repository but this is not very flexible (as for now you cannot even specify a git branch when deploying from python).

Another solution is to transfer the source code via the S3 bucket using prefect blocks.

In [ ]:
# Use a subfolder named after the current user
s3_folder = f"users/{getpass.getuser()}" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code and wheel to: '{S3_BLOCK.basepath}/{s3_folder}'")

# Upload local directory contents
await S3_BLOCK.put_directory(local_path = ".", to_path = s3_folder)

# It doesn't follow symlinks so upload them manually
await S3_BLOCK.put_directory(local_path = "./resources", to_path = f"{s3_folder}/resources")

In [ ]:
# Deploy the flow
flow = await prefect.flow.from_source(
    source=S3_BLOCK, # NOTE: we must 'pip install s3fs' in the worker containers
    entrypoint=f"{s3_folder}/my_prefect.py:flow_show_stars",
)
await flow.deploy(
    name="deploy-python",
    work_pool_name=PREFECT_WORK_POOL,
    tags=["tutorial"],
    ignore_warnings=True,
)
deploy_python = "flow-show-stars/deploy-python"
await my_prefect.wait_for_deployment(deploy_python)

In [ ]:
%%bash -s "$deploy_python" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2"

In [ ]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

<div class="alert alert-info" role="alert">
Notes:

  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.
  1. Check in the run logs that the client IP address is **different** than the one used by the Prefect flow and tasks.
  1. NOTE: you can open a terminal in the worker containers and check their IP address with:

      `python -c "import socket; print(socket.gethostbyname(socket.gethostname()))"`

### Define deployments with YAML
See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/prefect-yaml

Use YAML to schedule and trigger flow runs and manage your code and deployments.

This is the most complete way to deploy your flow and tasks.

**Deploy with YAML from git repository**

See the full yaml file: [deploy-yaml-git.yaml](./deploy-yaml-git.yaml)

As above, we need to store and transfer our flow and tasks from our local source code, saved in the project git repository. We can tell prefect to pull the source code from there before deploying it. 

This is the easiest way to transfer your code, but you must be careful to deploy your right git branch name, and push your code to git after each local modification.
```yaml
pull:
- prefect.deployments.steps.git_clone:
    repository: https://github.com/org/repo.git
    branch: main
    credentials: "{{ prefect.blocks.github-credentials.my-credentials }}"
```

We can install additional modules before running the flow either with:
```yaml
pull:
- prefect.deployments.steps.git_clone:
    id: clone-step # needed to be referenced in subsequent steps
    repository: https://github.com/org/repo.git
- prefect.deployments.steps.pip_install_requirements:
    directory: "{{ clone-step.directory }}" # `clone-step` is a user-provided `id` field
    requirements_file: requirements.txt
```
or:
```yaml
- prefect.deployments.steps.run_shell_script:
    script: pip install # ...
```
In this example, we will install the module `argh`.

NOTES:

  1. These additional installations will persist in the worker container, which means that they will still be there for other runs or deployments.
  1. We need to trigger the deployment from the git root folder and use the relative path to the yaml file. The entrypoint from the yaml file must also use a relative file from the git root folder:

```yaml
deployments:
- entrypoint: ./notebooks/tutorials/prefect+dask/my_prefect.py:flow_show_stars
```

In [ ]:
%%bash
# Go to the git root folder
cd ../../..
# Deploy the flow
prefect --no-prompt deploy --prefect-file "notebooks/tutorials/prefect+dask/deploy-yaml-git.yaml"

In [ ]:
deploy_yaml_git = "flow-show-stars/deploy-yaml-git"
await my_prefect.wait_for_deployment(deploy_yaml_git)

In [ ]:
%%bash -s "$deploy_yaml_git" "$my_data_str"
# Trigger a run for this flow from the command line. Test that the 'argh' module was installed.
prefect deployment run "$1" --param github_repos="$2" --param test_pip="argh"

**Deploy with YAML from S3 bucket**

See the full yaml file: [deploy-yaml-s3.yaml](./deploy-yaml-s3.yaml)

As when deploying flows with Python, we can transfer our local source code via the S3 bucket using prefect blocks. We can also transfer additional wheel files to be installed before running the flow. In this example, we will install the module `emoji`:

```yaml
pull:
- prefect.deployments.steps.run_shell_script:
    script: |
        python -c "from prefect.filesystems import RemoteFileSystem; RemoteFileSystem.load('s3').get_directory('users/jovyan', '.')"        
        pip install emoji --find-links ./wheels
```

NOTE: here also, these additional installations will persist in the worker container, which means that they will still be there for other runs or deployments.


In [ ]:
# Use a subfolder named after the current user
s3_folder = f"users/{getpass.getuser()}" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code and wheel to: '{S3_BLOCK.basepath}/{s3_folder}'")

# Upload local directory contents
await S3_BLOCK.put_directory(local_path = ".", to_path = s3_folder)

# It doesn't follow symlinks so upload them manually
await S3_BLOCK.put_directory(local_path = "./resources", to_path = f"{s3_folder}/resources")

# Do the same with a wheel file. First download it.
whl_dir = "/tmp/emoji"
!rm -rf $whl_dir && mkdir -p $whl_dir && pip download --dest $whl_dir emoji

# Then upload its directory to S3
await S3_BLOCK.put_directory(local_path = whl_dir, to_path = s3_folder + "/wheels")

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./deploy-yaml-s3.yaml"

In [ ]:
deploy_yaml_s3 = "flow-show-stars/deploy-yaml-s3"
await my_prefect.wait_for_deployment(deploy_yaml_s3)

In [ ]:
%%bash -s "$deploy_yaml_s3" "$my_data_str"
# Trigger a run for this flow from the command line. Test that the 'emoji' module was installed.
prefect deployment run "$1" --param github_repos="$2" --param test_pip="emoji"